In [2]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xar
import os
os.chdir("/home/philbou/projects/def-rfajber/philbou/analysis_paper1")

import diagnostic_plot_helper as dps  

In [3]:
# Only do once:
# Splits the datasets into managable chunks what we need 
#np.array([0.5,1,1.25,1.5,2])

ds_dic = {}

for alpha in np.array([1,1.25]):
    
    exp_folder_name = f"fr_T42_noconv_alpha{alpha}"
    start_file = 216
    end_file = 240

    #dps.split_save_exp(exp_folder_name,exp_folder_name,start_file,end_file)          

In [4]:
alpha_values = np.array([0.75,1,1.25])
ds_dict = {}

plot_path = "../plots/fr_T42_clim_exp_noconv/"
for alpha in alpha_values:
    path_folder = "/home/philbou/projects/def-rfajber/philbou/saved_ds"
    tmp = "fr_T42_clim_exp_noconv"

    # Create folder to save plots 
    folder_name_plots = "/home/philbou/projects/def-rfajber/philbou/plots/"+tmp

    os.makedirs(folder_name_plots, exist_ok=True)
    exp_folder_name = f"fr_T42_noconv_alpha{alpha}"
    #exp_folder_name = f"fr_T42_alpha{alpha}"
    path = f"{path_folder}/{exp_folder_name}"
    print(path)           
    # Load the age dataset for each alpha
    ds_tmp = xar.open_dataset(f"{path}/age.nc")
    ds_dyn = xar.open_dataset(f"{path}/dynamics.nc")
    ds_mix = xar.open_dataset(f"{path}/mixed_layer.nc")
    ts = ds_mix.t_surf
    tmp_dic = {
        'alpha' : alpha,
        "ds" : ds_tmp,
        "ds_dyn" : ds_dyn,
        "ts" : ts,
    }
    ds_dict[str(alpha)] = tmp_dic

# So we have a dict where we can access each alpha experiment's dataset
# We will calculate the global W, P, Tau, Pstar values and save then in this dict for easy access
# Then, we will make a plot of these values.

/home/philbou/projects/def-rfajber/philbou/saved_ds/fr_T42_noconv_alpha0.75


FileNotFoundError: [Errno 2] No such file or directory: '/home/philbou/projects/def-rfajber/philbou/saved_ds/fr_T42_noconv_alpha0.75/age.nc'

In [ ]:
# Define useful functions

def vertical_int(f,phalf,p_bot,p_top):
    # f is of the shape (time, pfull, lon, lat)
    dp = phalf[1:]-phalf[:-1]
    s = f.shape
    sum_integral = np.zeros((s[0],s[2],s[3]))
    for index in range(len(dp)):  # Include current index in sum
        fdp =  f[:,index,:,:] * dp[index]
        sum_integral += fdp
    return sum_integral

def vertical_rhow_avg(f,phalf,p_bot,p_top):
    frho_int = vertical_int(f,phalf,p_bot,p_top)
    den = 100000 #vertical_int(np.ones_like(f),phalf,p_bot,p_top)
    return frho_int/den

def area_w_avg(area,t_surf):
    temp = 0
    A = 0
    nlat = area.shape[0]
    nlon = area.shape[1]
    for i in range(nlat):
        for j in range(nlon):
            a = area[i][j]
            t = t_surf[i][j]
            temp += a*t
            A += a
    glob_avg_temp = temp/A
    return glob_avg_temp

def get_area(tmp_ds):
    # Calculate the area 

    lat = np.deg2rad(tmp_ds.lat.values)
    latb = tmp_ds.latb.values
    lon = tmp_ds.lon.values
    lonb = tmp_ds.lonb.values

    dlat = np.deg2rad(latb[1:] - latb[:-1])
    dlon = np.deg2rad(lonb[1:] - lonb[:-1])
    R = 6371
    area = np.zeros((len(lat),len(lon)))

    for i in range(len(lat)):
        for j in range(len(lon)):
            dlat_tmp = dlat[i]
            dlon_tmp = dlon[j]
            lat_tmp = lat[i]

            tmp = R**2 * np.cos(lat_tmp) * dlon_tmp * dlat_tmp

            area[i][j] = abs(tmp)
    return area

In [ ]:
def glob_hyd_budget(ds,ds_dyn,ts):
    p_bot = ds.pfull[-1]
    p_top = ds.pfull[0]
    phalf = 100*ds.phalf.values
    sink = ds.dt_sink
    tau = ds.sphum_age_1
    q = ds.sphum.values
    z0 = np.zeros((ds.height.shape[0], 1, ds.height.shape[2], ds.height.shape[3]))
    z = np.append(z0, ds.height.values, axis=1)
    g = 9.81
    dP = 100*(ds.phalf[1:].values - ds.phalf[:-1].values)
    dP_reshaped = dP[np.newaxis, :, np.newaxis, np.newaxis]
    dz = z[:,1:,:,:] - z[:,:-1,:,:]
    rho = (-1/g) * dP_reshaped/dz
    col_air = ds.ps.values/g # kg/m^2
    vert_rho = vertical_int(rho,phalf,p_bot,p_top)

    # P is kg s^-1 but I want kg/kg s^-1
    P = ds.precipitation.values/col_air
    
    area = get_area(ds)
    
    C = -sink.values
    vert_C = vertical_rhow_avg(C,phalf,p_bot,p_top)
    time_vert_C = np.mean(vert_C,axis = 0)
    Pstar = vert_C - P
    time_Pstar = np.mean(Pstar,axis = 0)
    
    W = vertical_rhow_avg(q,phalf,p_bot,p_top)
    vert_tau = vertical_rhow_avg(tau.values,phalf,p_bot,p_top)
    time_W = np.mean(W,axis = 0)
    time_P = np.mean(P,axis = 0)
    time_ts = np.mean(ts.values,axis = 0)
    glob_ts = area_w_avg(area,time_ts)
    
    
    time_vert_tau = np.mean(vert_tau,axis = 0)
    glob_W = area_w_avg(area,time_W)
    glob_P = area_w_avg(area,time_P)
    glob_Pstar = area_w_avg(area,time_Pstar)
    glob_tau = area_w_avg(area,time_vert_tau)
    return glob_W,glob_P,glob_Pstar, glob_tau,glob_ts
    

$$\frac{\langle W \rangle}{\langle P \rangle} = \frac{\langle \{\Lambda\} \rangle}{\langle W \rangle} \left( 1 + \frac{\langle P^* \rangle}{\langle P \rangle} \right) + \frac{\langle R \rangle}{\langle W \rangle}$$

In [ ]:
def calc_glob_budget(ds_dict,box_top,box_sides,alpha_values,output = False):
    
    for alpha in alpha_values:
        tmp_dict = ds_dict[str(alpha)]
        ds_ = tmp_dict["ds"]
        ds_dyn_ = tmp_dict["ds_dyn"]

        if box_sides[0] < box_sides[1]:
            # Contiguous latitude range
            ds_dyn = ds_dyn_.sel(
                lat=slice(box_sides[0], box_sides[1]), 
                pfull=slice(box_top[0], box_top[1]),
                phalf=slice(box_top[0], box_top[1])
            )

            ds = ds_.sel(
                lat=slice(box_sides[0], box_sides[1]), 
                pfull=slice(box_top[0], box_top[1]),
                phalf=slice(box_top[0], box_top[1])
            )
        else:
            # Non-contiguous case, e.g., poleward of 60°
            ds_dyn = ds_dyn_.where(
                (ds_dyn_.lat >= box_sides[0]) | (ds_dyn_.lat <= box_sides[1]),
                drop=True
            ).sel(pfull=slice(box_top[0], box_top[1])).sel(phalf=slice(box_top[0], box_top[1]))

            ds = ds_.where(
                (ds_.lat >= box_sides[0]) | (ds_.lat <= box_sides[1]),
                drop=True
            ).sel(pfull=slice(box_top[0], box_top[1])).sel(phalf=slice(box_top[0], box_top[1]))

        ts = tmp_dict["ts"]
        glob_W,glob_P,glob_Pstar, glob_tau,glob_ts = glob_hyd_budget(ds,ds_dyn,ts)
        tmp_dict["W"] = glob_W
        tmp_dict["P"] = glob_P
        tmp_dict["Pstar"] = glob_Pstar
        tmp_dict["tau"] = glob_tau
        tmp_dict["glob_ts"] = glob_ts

        term1 = glob_W/glob_P
        term2 = glob_tau/glob_W
        term3 = term2 * glob_Pstar/glob_P
        term4 = term1-(term2+term3)

        tmp_dict["W/P"] = term1/(24*60**2)
        tmp_dict["tau/W"] = term2/(24*60**2)
        tmp_dict["star"] = term3/(24*60**2)
        tmp_dict["remainder"] = term4/(24*60**2)
        if output:
            print("alpha = ", alpha)
            print("W/P",term1/(24*60**2))
            print("tau/W",term2/(24*60**2))
            print("star",term3/(24*60**2))
            print("remainder",term4/(24*60**2))
    return ds_dict
    
def get_frac_inc_rate(arr,ts):
    dts = np.gradient(ts)
    darr =np.gradient(arr)/arr
    return 100*(darr/dts)

def make_arrays_hyd_sens(ds_dict,alpha_values):
    
    Ws = []
    Ps = []
    taus = []
    tauWs = []
    stars = []
    rems = []
    As = []

    tsvals = []

    for alpha in alpha_values:
        tmp_dict = ds_dict[str(alpha)]
        W = tmp_dict["W"]
        P = tmp_dict["P"]
        tauW = tmp_dict["tau/W"]
        A = tmp_dict["tau"]
        tau = tmp_dict["W/P"]
        tsval = tmp_dict["glob_ts"]
        star = tmp_dict["star"]
        remainder =  tmp_dict["remainder"]
        Ws.append(W)
        As.append(A)
        Ps.append(P)
        taus.append(tau)
        tauWs.append(tauW)
        stars.append(star)
        rems.append(remainder)
        tsvals.append(tsval)



    Ws = np.array(Ws)
    As = np.array(As)
    Ps = np.array(Ps)
    taus = np.array(taus)
    tauWs = np.array(tauWs)
    stars = np.array(stars)
    rems = np.array(rems)
    tsvals = np.array(tsvals)
    
    return Ws,As,Ps,taus,tauWs,stars,rems,tsvals


In [ ]:

def plot_hyd_sens(ds_dict,alpha_values,name_ext):
    Ws,As,Ps,taus,tauWs,stars,rems,tsvals = make_arrays_hyd_sens(ds_dict,alpha_values)
    print(Ws)
    print(Ps)
    print(As)
    dT = tsvals[-1]- tsvals[0]
    fig,ax = plt.subplots(1,1, figsize = (7,5))

    ax.scatter(tsvals,tauWs,marker = ".",color = "mediumblue",label = r"$\frac{\langle \Lambda \rangle}{\langle W \rangle}$")
    #ax.plot(tsvals, ltauW + dtauW * tsvals,color = "mediumblue")
    props = dict(boxstyle='square',pad=0.2, facecolor='white', alpha=1,edgecolor = "mediumblue")
    ax.text(280,np.max(tauWs), f"{round(100*(tauWs[-1]-tauWs[0])/(np.mean(tauWs)*dT),2)} %", fontsize=14,
            verticalalignment='top', bbox=props)


    ax.scatter(tsvals,taus,marker = ".",color = "darkorange",label = r"$\frac{\langle W \rangle}{\langle P \rangle}$")
    #ax.plot(tsvals, ltau + dtau * tsvals,color = "darkorange")
    props = dict(boxstyle='square',pad=0.2, facecolor='white', alpha=1,edgecolor = "darkorange")
    ax.text(280,np.max(taus), f"{round(100*(taus[-1]-taus[0])/(np.mean(taus)*dT),2)} %", fontsize=14,
            verticalalignment='top', bbox=props)


    ax.scatter(tsvals,rems,marker = ".",color = "violet",label = r"$\frac{\langle R \rangle}{\langle W \rangle}$")
    #ax.plot(tsvals, lrem + drem * tsvals,color = "violet")
    props = dict(boxstyle='square',pad=0.2, facecolor='white', alpha=1,edgecolor = "violet")
    ax.text(280,0.5*np.min(rems), f"{round(100*(rems[-1]-rems[0])/(np.mean(rems)*dT),2)} %", fontsize=14,
            verticalalignment='top', bbox=props)


    ax.scatter(tsvals,stars,marker = ".",color = "green",label = r"$\frac{\langle \{\Lambda\} \rangle}{\langle W \rangle}\frac{\langle P^* \rangle}{\langle P \rangle}$")
    #ax.plot(tsvals, lstar + dstar * tsvals,color = "green")
    props = dict(boxstyle='square',pad=0.2, facecolor='white', alpha=1,edgecolor = "green")
    ax.text(280,0.8*np.max(stars), f"{round(100*(stars[-1]-stars[0])/(np.mean(stars)*dT),2)} %", fontsize=14,
            verticalalignment='top', bbox=props)

    ax.set_xlabel(r"$\langle T_s \rangle$")
    fig.suptitle(name_ext)
    top = np.max(tauWs) + 5
    bot = np.min(rems) - 5
    ax.set_ylim(bot,top)
    ax.set_ylabel("Days")
    ax.legend(
    ncol=2,
    loc='center left',
    bbox_to_anchor=(1.05, 0.5),
    frameon=True)
    fig.savefig(plot_path + f"hyd_sens_{name_ext}.png",dpi = 300,bbox_inches='tight')

In [ ]:
ds_dict_t = calc_glob_budget(ds_dict,(0,1000),(-90,90),alpha_values)
plot_hyd_sens(ds_dict_t,alpha_values,"test")

In [ ]:
"""lat_regions_arr = [(-90,90),(-60,60),(-30,30),(60,-60)]
p_int_arr = [(0,1000),(300,1000), (450,1000),(500,1000)]

for lat_int in lat_regions_arr:
    for p_int in p_int_arr:
        print(p_int,lat_int)
        name = f"P:({p_int[0]},{p_int[1]})_L:({lat_int[0]},{lat_int[1]})"
        ds_dict_tmp = calc_glob_budget(ds_dict,(p_int[0],p_int[1]),(lat_int[0],lat_int[1]),alpha_values)
        plot_hyd_sens(ds_dict_tmp,alpha_values,name)"""

Now, we want to look at the difference between the runs

In [ ]:
ds_dict = calc_glob_budget(ds_dict,(0,1000),(-90,90),alpha_values)
Ws,As,Ps,taus,tauWs,stars,rems,tsvals = make_arrays_hyd_sens(ds_dict,alpha_values)
dT = tsvals[-1]- tsvals[0]
mean_arr = []
std_arr = []
ps_arr = []
pot_temp_zonal_arr = []
for alpha in alpha_values:
    tmp_dict = ds_dict[str(alpha)]
    ds = tmp_dict["ds"]
    ds_dyn = tmp_dict["ds_dyn"]
    temp = ds_dyn.temp
    ps = ds_dyn.ps
    phalf = ds_dyn.phalf.values
    lat = ds_dyn.lat.values
    lon = ds_dyn.lon.values

    pfull = ds_dyn.pfull.values
    sphum = ds.sphum
    qmoments = [ds.sphum_age_1,ds.sphum_age_2]#,ds_age.sphum_age_3,ds_age.sphum_age_4]
    moments = [qmoments[0]/sphum,qmoments[1]/sphum]#qmoments[2]/sphum,qmoments[3]/sphum]
    central_moments = [moments[0], moments[1]-moments[0]**2]
    std = np.sqrt(central_moments[1])
    mean = central_moments[0]
    
    pot_temp = dps.get_pot_temp(temp,pfull,ps)
    pot_temp_zonal = np.mean(np.mean(pot_temp,axis = 0),axis = 2)
    pot_temp_zonal_arr.append(pot_temp_zonal)
    
    ps_arr.append(ps)
    if alpha == 1:#1:
        mean1= mean
        std1 = std
    else:
        std_arr.append(std)
        mean_arr.append(mean)
    

In [ ]:
print("P:", f"{round(100*(Ps[-1]-Ps[0])/(np.mean(Ps)*dT),2)} %")
print("W:", f"{round(100*(Ws[-1]-Ws[0])/(np.mean(Ws)*dT),2)} %")

In [ ]:
i=0 
j=0
for alpha in alpha_values:
    if alpha == 1:
        fig,axs = plt.subplots(1,figsize = (7,5))
        fig,ax = dps.plot_age_moments_vertical_profile(21,mean1,1,ps_arr[i],pfull,lat,title = f"Ts = {round(tsvals[j],0)}",pot_temp = pot_temp_zonal_arr[i],central=True,cmap = "gist_ncar",time_avg = True,lnP=False,custom = True, ax = axs)
    else:
        fig,axs = plt.subplots(1,2,figsize = (15,5))
        fig,ax = dps.plot_age_moments_vertical_profile(21,mean_arr[i],1,ps_arr[i],pfull,lat,title = f"Ts = {round(tsvals[j],0)}",pot_temp = pot_temp_zonal_arr[i],central=True,cmap = "gist_ncar",time_avg = True,lnP=False,custom = True, ax = axs[0])

        diff = mean_arr[i].mean(dim = ["time","lon"]) - mean1.mean(dim = ["time","lon"])

        if alpha == 1.75:
            dddd = diff
        max_val = 14#np.max(abs(diff))
        steps = 2*max_val+1
        ax = axs[1]
        dps.plot_vertical_profile(diff/(24*60*60),lat,pfull,axs[1],"seismic",level_space=np.linspace(-max_val,max_val,steps),lnP = False,extend = "neither")
        isentropic_levels = np.arange(280, 400, 10)
        cs=ax.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Anomally wrt Ts = {round(tsvals[2],0)}",fontsize = 25)
        i+=1
    plt.savefig(plot_path + f"age_Ts{round(tsvals[j],0)}.png",dpi = 300)
    j+=1


In [ ]:
i=0 
j=0
for alpha in alpha_values:
    if alpha == 1:
        fig,axs = plt.subplots(1,figsize = (7,5))
        fig,ax = dps.plot_shape_param_vertical_profile(std1,mean1,pfull,lat,ps_arr[i],pot_temp_zonal_arr[i],central=True,cmap = "jet",time_avg = True,lnP=False,custom = True, ax = axs)
        title = f"Ts = {round(tsvals[j],0)}"
        ax.set_title(title,fontsize = 25)
    else:
        fig,axs = plt.subplots(1,2,figsize = (15,5))
        title = f"Ts = {round(tsvals[j],0)}"
        ax = axs[0]
        fig,ax = dps.plot_shape_param_vertical_profile(std_arr[i],mean_arr[i],pfull,lat,ps_arr[i],pot_temp_zonal_arr[i],central=True,cmap = "jet",time_avg = True,lnP=False,custom = True, fig = fig,ax = ax)
        ax.set_title(title,fontsize = 25)
        s_a = ((mean_arr[i]/std_arr[i])**(1.086)).mean(dim = ["time","lon"])
        s1 = ((mean1/std1)**(1.086)).mean(dim = ["time","lon"])
        diff = s_a - s1
        max_val = 2#np.max(abs(diff))
        steps = 41
        ax = axs[1]
        dps.plot_vertical_profile(diff,lat,pfull,ax,"seismic",level_space=np.linspace(-max_val,max_val,steps),lnP = False,extend = "max")
        isentropic_levels = np.arange(280, 400, 10)
        cs=ax.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Anomally wrt Ts = {round(tsvals[2],0)}",fontsize = 25)
        i+=1
    plt.savefig(plot_path + f"shape_Ts{round(tsvals[j],0)}.png",dpi = 300)
    j+=1

In [ ]:
omegap2_arr = []
temp_arr = []
for alpha in alpha_values:
    tmp_dict = ds_dict[str(alpha)]
    ds = tmp_dict["ds"]
    ds_dyn = tmp_dict["ds_dyn"]
    omega = ds_dyn.omega.mean(dim = "lon")
    omega_mean = omega.mean(dim = "time")
    omegaprime = omega- omega_mean
    omegaprime2 = (omegaprime**2).mean(dim = "time")
    temp = ds_dyn.temp.mean(dim = ["time","lon"])
    
    if alpha == 1:
        omegap21= omegaprime2
        temp1 = temp
    else:    
        omegap2_arr.append(omegaprime2)
        temp_arr.append(temp)
    

In [ ]:
i=0 
j=0
max_val = np.max(abs(omegap21))
steps = 51
for alpha in alpha_values:
    if alpha == 1:
        fig,axs = plt.subplots(1,figsize = (12,5))
        
        
        dps.plot_vertical_profile(omegap21,lat,pfull,axs,"seismic",level_space=np.linspace(-max_val,max_val,steps),lnP = False,extend = "max")
        isentropic_levels = np.arange(280, 400, 10)
        cs=axs.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Ts = {round(tsvals[1],0)}",fontsize = 25)
    else:
        fig,axs = plt.subplots(1,2,figsize = (25,5))
        dps.plot_vertical_profile(omegap2_arr[i],lat,pfull,axs[0],"seismic",level_space=np.linspace(-max_val,max_val,steps),lnP = False,extend = "max")
        isentropic_levels = np.arange(280, 400, 10)
        cs=axs[0].contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        axs[0].set_title(f"Ts = {round(tsvals[j],0)}",fontsize = 25)
        
        diff = (omegap2_arr[i] - omegap21)

        ax = axs[1]
        dps.plot_vertical_profile(diff,lat,pfull,axs[1],"seismic",level_space=np.linspace(-max_val,max_val,steps),lnP = False,extend = "neither")
        isentropic_levels = np.arange(280, 400, 10)
        cs=ax.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Anomally w.r.t Ts = {round(tsvals[2],0)}",fontsize = 25)
        i+=1
    plt.savefig(plot_path + f"omega_Ts{round(tsvals[j],0)}.png",dpi = 300)
    j+=1

In [ ]:
i=0 
j=0
for alpha in alpha_values:
    if alpha == 1:
        fig,axs = plt.subplots(1,figsize = (7,5))
        max_val = np.max(abs(temp1))
        steps = 50
        dps.plot_vertical_profile(temp1,lat,pfull,axs,"gist_ncar",level_space=np.linspace(200,max_val,steps),lnP = False,extend = "max")
        isentropic_levels = np.arange(280, 400, 10)
        cs=axs.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Ts = {round(tsvals[1],0)}",fontsize = 25)
    else:
        fig,axs = plt.subplots(1,2,figsize = (15,5))
        
        max_val = np.max(abs(temp1))
        steps = 50
        dps.plot_vertical_profile(temp_arr[i],lat,pfull,axs[0],"gist_ncar",level_space=np.linspace(200,max_val,steps),lnP = False,extend = "max")
        isentropic_levels = np.arange(280, 400, 10)
        cs=axs[0].contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        axs[0].set_title(f"Ts = {round(tsvals[j],0)}",fontsize = 25)
        
        diff = (temp_arr[i] - temp1)

        ax = axs[1]
        dps.plot_vertical_profile(diff,lat,pfull,axs[1],"seismic",level_space=np.linspace(-50,50,41),lnP = False,extend = "neither")
        isentropic_levels = np.arange(280, 400, 10)
        cs=ax.contour(lat, pfull, pot_temp_zonal_arr[i],colors="black",levels = isentropic_levels)
        plt.clabel(cs, inline=True, fontsize=14)
        ax.set_title(f"Anomally w.r.t Ts = {round(tsvals[2],0)}",fontsize = 25)
        i+=1
    plt.savefig(plot_path + f"temp_Ts{round(tsvals[j],0)}.png",dpi = 300)
    j+=1

In [ ]:
# Age of precip

